# LeetCode 2007 — Find Original Array From Doubled Array bằng Python

Link đề: https://leetcode.com/problems/find-original-array-from-doubled-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findOriginalArray(self, changed: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Hiểu khái niệm **đa tập** (multiset) và cách so sánh hai đa tập bằng `Counter` hoặc `sorted`.
2. Phát hiện **quan sát tham lam**: phần tử **nhỏ nhất** còn lại **bắt buộc** thuộc mảng gốc, nên phải ghép nó với giá trị gấp đôi.
3. Cài đặt tham lam bằng **sắp xếp + `Counter`** (`O(n log n)`), bằng **mảng đếm** (`O(n + V)`), và bằng **hàng đợi chờ ghép**.
4. Xử lý các trường hợp đặc biệt: độ dài lẻ, **số 0** (gấp đôi của 0 là chính nó), phần tử lặp.
5. Kiểm thử bài **nhiều đáp án** (trả về theo thứ tự bất kỳ) bằng **hàm kiểm tra hợp lệ** và đối chiếu brute force.

## 2. Đề bài & ví dụ

Một mảng `original` được biến thành mảng **gấp đôi** `changed` bằng cách: thêm vào mảng **hai lần giá trị** của mọi phần tử (`2·x` cho mỗi `x`), rồi **xáo trộn** ngẫu nhiên.

Cho `changed`, trả về `original` nếu `changed` là một mảng gấp đôi; ngược lại trả về **mảng rỗng**. Các phần tử của `original` có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1:** `changed = [1,3,4,2,6,8]` → `[1,3,4]` (vì `2 = 2·1`, `6 = 2·3`, `8 = 2·4`). `[4,3,1]` hay `[3,1,4]` cũng đúng.

**Ví dụ 2:** `changed = [6,3,0,1]` → `[]`.

**Ví dụ 3:** `changed = [1]` → `[]`.

**Ràng buộc:** `1 ≤ len(changed) ≤ 10⁵`, `0 ≤ changed[i] ≤ 10⁵`.

In [1]:
import random
import time
from collections import Counter, deque
from itertools import combinations
from typing import List

examples = [([1, 3, 4, 2, 6, 8], [1, 3, 4]), ([6, 3, 0, 1], []), ([1], [])]
for ch, exp in examples:
    print(f"changed={ch} -> một đáp án đúng: {exp}")

changed=[1, 3, 4, 2, 6, 8] -> một đáp án đúng: [1, 3, 4]
changed=[6, 3, 0, 1] -> một đáp án đúng: []
changed=[1] -> một đáp án đúng: []


## 3. Kiến thức nền

### 3.1 Đa tập và `Counter`

Thứ tự không quan trọng nhưng **số lần lặp** thì quan trọng: `[1, 2, 2]` và `[2, 1, 2]` là cùng đa tập, còn `[1, 2]` thì khác. `Counter(a) == Counter(b)` hoặc `sorted(a) == sorted(b)` so sánh hai đa tập.

In [2]:
print(Counter([1, 2, 2]) == Counter([2, 1, 2]), Counter([1, 2, 2]) == Counter([1, 2]))
print(sorted([2, 1, 2]) == sorted([1, 2, 2]))
c = Counter([1, 3, 4, 2, 6, 8]); c[2] -= 1
print("sau khi dùng một số 2:", c)

True False
True
sau khi dùng một số 2: Counter({1: 1, 3: 1, 4: 1, 6: 1, 8: 1, 2: 0})


### 3.2 Hàm kiểm tra một đáp án có hợp lệ không

Vì đề cho phép trả về theo thứ tự bất kỳ, ta không so sánh trực tiếp với một mảng mẫu mà dùng: `original` hợp lệ ⇔ `Counter(original + [2x for x in original]) == Counter(changed)`.

In [3]:
def is_doubled_of(changed, original):
    return Counter(original + [2 * x for x in original]) == Counter(changed)

print(is_doubled_of([1, 3, 4, 2, 6, 8], [4, 1, 3]), is_doubled_of([1, 3, 4, 2, 6, 8], [1, 2, 3]))

True False


### 3.3 Sắp xếp và duyệt khóa của `Counter` theo thứ tự tăng

`sorted(cnt)` trả về các **khóa khác nhau** theo thứ tự tăng — ít phần tử hơn `sorted(changed)` khi có nhiều giá trị lặp.

In [4]:
cnt = Counter([4, 2, 2, 1, 8, 4])
print("các khóa tăng dần:", sorted(cnt), "| tần suất:", [cnt[k] for k in sorted(cnt)])

các khóa tăng dần: [1, 2, 4, 8] | tần suất: [1, 2, 2, 1]


### 3.4 Mảng đếm (counting) khi giá trị nhỏ

Khi `0 ≤ giá trị ≤ V` với `V` vừa phải (`10⁵`), có thể dùng `cnt = [0] * (V + 1)` thay cho dict — duyệt `x` từ `0` đến `V` là đã "sắp xếp" sẵn.

In [5]:
V = 10
cnt = [0] * (V + 1)
for x in [1, 3, 4, 2, 6, 8]:
    cnt[x] += 1
print([(x, cnt[x]) for x in range(V + 1) if cnt[x]])

[(1, 1), (2, 1), (3, 1), (4, 1), (6, 1), (8, 1)]


## 4. Ý tưởng nền tảng / trực giác

Nhìn `changed = [1,3,4,2,6,8]` sau khi sắp xếp: `[1,2,3,4,6,8]`.

- Số **nhỏ nhất** là `1`. Nó có thể là "gấp đôi của ai"? Chỉ của `0.5` — không có. Vậy `1` **phải** thuộc `original`, và `2` phải có mặt để làm đôi của nó. Ghép `(1, 2)`.
- Còn `[3,4,6,8]`. Nhỏ nhất `3` ⇒ ghép với `6`. Còn `[4,8]` ⇒ ghép `(4, 8)`.
- Hết ⇒ `original = [1,3,4]`.

`changed = [6,3,0,1]` → sắp `[0,1,3,6]`: `0` cần một `0` khác để ghép — không có ⇒ `[]`.

**Quan sát:** phần tử nhỏ nhất (dương) không thể là "đôi" của một phần tử khác còn lại (vì đôi của nó ở trong mảng phải nhỏ hơn nó). Đây là chìa khóa tham lam.

In [6]:
arr = sorted([1, 3, 4, 2, 6, 8])
pairs = []
while arr:
    x = arr.pop(0)            # nhỏ nhất
    arr.remove(2 * x)         # ở ví dụ này luôn có
    pairs.append((x, 2 * x))
    print("ghép", (x, 2 * x), "| còn lại:", arr)
print("original =", [p[0] for p in pairs])

ghép (1, 2) | còn lại: [3, 4, 6, 8]
ghép (3, 6) | còn lại: [4, 8]
ghép (4, 8) | còn lại: []
original = [1, 3, 4]


## 5. Cách 1 — Brute Force / baseline: thử mọi tập con kích thước n/2

**Ý tưởng:** nếu `len(changed)` lẻ ⇒ `[]`. Ngược lại thử mọi cách chọn `n/2` chỉ số làm `original`, kiểm tra phần còn lại có đúng là các giá trị gấp đôi không.

In [7]:
def find_brute(changed):
    n = len(changed)
    if n % 2:
        return []
    for idx in combinations(range(n), n // 2):
        orig = [changed[i] for i in idx]
        rest = [changed[i] for i in range(n) if i not in set(idx)]
        if Counter(2 * x for x in orig) == Counter(rest):
            return orig
    return []

for ch, exp in examples:
    got = find_brute(ch)
    print(ch, "->", got, "| hợp lệ:", is_doubled_of(ch, got) if got else got == exp)

[1, 3, 4, 2, 6, 8] -> [1, 3, 4] | hợp lệ: True
[6, 3, 0, 1] -> [] | hợp lệ: True
[1] -> [] | hợp lệ: True


**Độ phức tạp:** `C(n, n/2) ≈ 2ⁿ / √n` tổ hợp, mỗi tổ hợp `O(n)` ⇒ hàm mũ. Chỉ dùng được với `n ≤ ~16`. Space `O(n)`.

**Nút thắt:** ta thử mọi cách chọn trong khi quan sát ở mục 4 cho thấy mỗi bước chỉ có **một** lựa chọn hợp lý: phần tử nhỏ nhất phải là phần tử gốc.

In [8]:
for n in (8, 12, 16):
    ch = list(range(1, n // 2)) + [2 * x for x in range(1, n // 2)] + [7, 9]   # gần đúng nhưng hỏng ở cuối
    t = time.perf_counter(); find_brute(ch)
    print(f"n={len(ch)}: {time.perf_counter() - t:.3f}s")

n=8: 0.000s
n=12: 0.004s
n=16: 0.059s


## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề tham lam:** gọi `m` là phần tử nhỏ nhất của đa tập còn lại.
- **Nếu** `m > 0` **thì** `m` không thể là "đôi" của phần tử `y` nào còn lại (cần `y = m/2 < m`, trái với tính nhỏ nhất) ⇒ `m` thuộc `original` ⇒ phải xóa một `m` và một `2m`. **Nếu** không có `2m` ⇒ trả `[]`.
- **Nếu** `m = 0` **thì** `0` ghép với `0`: số lượng số `0` phải **chẵn**, và một nửa trong số đó thuộc `original`.

**Invariant:** sau mỗi bước, đa tập còn lại là mảng gấp đôi **khi và chỉ khi** đa tập ban đầu là mảng gấp đôi (vì bước ghép là bắt buộc).

Xử lý theo giá trị tăng dần với `Counter`: với mỗi giá trị `x` (tăng dần) còn `cnt[x] > 0` bản, cần `cnt[2x] ≥ cnt[x]` (với `x > 0`), rồi `cnt[2x] -= cnt[x]` — xử lý cả nhóm một lúc.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + xóa khỏi danh sách (tham lam đơn giản, `O(n²)`)

Làm đúng như mục 4 nhưng có kiểm tra: lấy nhỏ nhất, tìm và xóa `2x`. `list.remove` là `O(n)` ⇒ tổng `O(n²)`.

In [9]:
def find_greedy_list(changed):
    if len(changed) % 2:
        return []
    arr = sorted(changed)
    res = []
    while arr:
        x = arr.pop(0)
        try:
            arr.remove(2 * x)
        except ValueError:
            return []
        res.append(x)
    return res

for ch, exp in examples:
    print(ch, "->", find_greedy_list(ch))

[1, 3, 4, 2, 6, 8] -> [1, 3, 4]
[6, 3, 0, 1] -> []
[1] -> []


**Độ phức tạp Cách 2:** `O(n²)` vì `pop(0)` và `remove` đều `O(n)`. Đủ đúng nhưng `n = 10⁵` thì chậm.

### 7.2 Cách 3 — Sắp xếp + `Counter` (bản nộp chính)

**Giải thích từng đoạn:**
1. Độ dài lẻ ⇒ `[]`.
2. `cnt = Counter(changed)`.
3. Duyệt `x` trong `sorted(changed)` (từng phần tử): nếu `cnt[x] == 0` (đã bị dùng làm "đôi") thì bỏ qua.
4. Giảm `cnt[x]`; nếu `cnt[2x] == 0` ⇒ `[]`; giảm `cnt[2x]`; thêm `x` vào kết quả.

Giảm `cnt[x]` **trước** khi kiểm `cnt[2x]` giúp trường hợp `x = 0` tự đúng (cần hai số 0).

In [10]:
def find_counter(changed):
    if len(changed) % 2:
        return []
    cnt = Counter(changed)
    res = []
    for x in sorted(changed):
        if cnt[x] == 0:
            continue
        cnt[x] -= 1
        if cnt[2 * x] == 0:
            return []
        cnt[2 * x] -= 1
        res.append(x)
    return res

for ch, exp in examples:
    got = find_counter(ch); print(ch, "->", got); assert got == exp

[1, 3, 4, 2, 6, 8] -> [1, 3, 4]
[6, 3, 0, 1] -> []
[1] -> []


In [11]:
class SolutionCounter:
    def findOriginalArray(self, changed: List[int]) -> List[int]:
        if len(changed) % 2:
            return []
        cnt = Counter(changed)
        res = []
        for x in sorted(changed):
            if cnt[x] == 0:
                continue
            cnt[x] -= 1
            if cnt[2 * x] == 0:
                return []
            cnt[2 * x] -= 1
            res.append(x)
        return res

print([SolutionCounter().findOriginalArray(ch) for ch, _ in examples])

[[1, 3, 4], [], []]


**Độ phức tạp Cách 3:** sắp xếp `O(n log n)`, vòng lặp `O(n)` (mỗi thao tác `Counter` `O(1)` trung bình) ⇒ `O(n log n)`. Space `O(n)`.

### 7.3 Cách 4 — Mảng đếm theo giá trị, xử lý cả nhóm (`O(n + V)`)

Vì `0 ≤ changed[i] ≤ 10⁵`, ta đếm vào mảng và duyệt `x = 0, 1, ..., V`:
- `x = 0`: cần `cnt[0]` chẵn; thêm `cnt[0] // 2` số 0.
- `x > 0`: `k = cnt[x]` bản `x` đều là gốc; cần `2x ≤ V` và `cnt[2x] ≥ k`; `cnt[2x] -= k`; thêm `k` bản `x`.

In [12]:
def find_counting(changed):
    if len(changed) % 2:
        return []
    V = max(changed)
    cnt = [0] * (V + 1)
    for x in changed:
        cnt[x] += 1
    if cnt[0] % 2:
        return []
    res = [0] * (cnt[0] // 2)
    for x in range(1, V + 1):
        k = cnt[x]
        if k == 0:
            continue
        if 2 * x > V or cnt[2 * x] < k:
            return []
        cnt[2 * x] -= k
        res.extend([x] * k)
    return res

for ch, exp in examples:
    got = find_counting(ch); print(ch, "->", got); assert got == exp

[1, 3, 4, 2, 6, 8] -> [1, 3, 4]
[6, 3, 0, 1] -> []
[1] -> []


In [13]:
class SolutionCounting:
    def findOriginalArray(self, changed: List[int]) -> List[int]:
        if len(changed) % 2:
            return []
        V = max(changed)
        cnt = [0] * (V + 1)
        for x in changed:
            cnt[x] += 1
        if cnt[0] % 2:
            return []
        res = [0] * (cnt[0] // 2)
        for x in range(1, V + 1):
            k = cnt[x]
            if k:
                if 2 * x > V or cnt[2 * x] < k:
                    return []
                cnt[2 * x] -= k
                res.extend([x] * k)
        return res

print([SolutionCounting().findOriginalArray(ch) for ch, _ in examples])

[[1, 3, 4], [], []]


**Độ phức tạp Cách 4:** `O(n + V)` thời gian, `O(V)` bộ nhớ, với `V = max(changed) ≤ 10⁵`.

### 7.4 Cách 5 — Sắp xếp + hàng đợi "đang chờ được ghép"

Duyệt mảng đã sắp. Hàng đợi `pending` chứa các giá trị `2x` đang chờ (theo thứ tự tăng vì `x` tăng). Với mỗi `y`: nếu `pending` khác rỗng và `pending[0] == y` thì `y` là "đôi" — lấy ra; ngược lại `y` là phần tử gốc, đẩy `2y` vào. Cuối cùng `pending` phải rỗng. Không cần băm!

In [14]:
def find_queue(changed):
    if len(changed) % 2:
        return []
    pending, res = deque(), []
    for y in sorted(changed):
        if pending and pending[0] == y:
            pending.popleft()
        else:
            if pending and pending[0] < y:     # có giá trị chờ mà không bao giờ tới
                return []
            res.append(y)
            pending.append(2 * y)
    return res if not pending else []

for ch, exp in examples:
    got = find_queue(ch); print(ch, "->", got); assert got == exp

[1, 3, 4, 2, 6, 8] -> [1, 3, 4]
[6, 3, 0, 1] -> []
[1] -> []


**Vì sao hàng đợi đúng:** `pending` luôn tăng dần (đẩy `2y` với `y` tăng). Nếu `y` bằng phần tử chờ nhỏ nhất, việc dùng `y` làm "đôi" là bắt buộc — nếu thay vào đó coi `y` là gốc thì giá trị chờ `pending[0] = y` sẽ không còn ai lấp (các phần tử sau đều `≥ y`, và nếu bằng `y` thì cùng tình huống). **Độ phức tạp:** `O(n log n)` (sắp xếp), `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên `changed = [4, 2, 2, 1, 8, 4]` (đáp án `[1, 2, 4]`). Sắp xếp: `[1, 2, 2, 4, 4, 8]`, `cnt = {1:1, 2:2, 4:2, 8:1}`.

| x | cnt[x] trước | hành động | cnt sau | res |
|---|---|---|---|---|
| 1 | 1 | ghép (1, 2) | {1:0, 2:1, 4:2, 8:1} | [1] |
| 2 | 1 | ghép (2, 4) | {2:0, 4:1, 8:1} | [1, 2] |
| 2 | 0 | bỏ qua (đã làm đôi) | | |
| 4 | 1 | ghép (4, 8) | {4:0, 8:0} | [1, 2, 4] |
| 4 | 0 | bỏ qua | | |
| 8 | 0 | bỏ qua | | |

In [15]:
def trace_counter(changed):
    if len(changed) % 2:
        print("độ dài lẻ -> []"); return []
    cnt = Counter(changed); res = []
    print("sắp xếp:", sorted(changed), "| cnt =", dict(cnt))
    for x in sorted(changed):
        if cnt[x] == 0:
            print(f"x={x}: cnt[x]=0 -> bỏ qua (đã dùng làm đôi)"); continue
        cnt[x] -= 1
        if cnt[2 * x] == 0:
            print(f"x={x}: không còn {2 * x} để ghép -> []"); return []
        cnt[2 * x] -= 1; res.append(x)
        print(f"x={x}: ghép ({x}, {2 * x}) | cnt={ {k: v for k, v in cnt.items() if v} } | res={res}")
    return res

print("Đáp án:", trace_counter([4, 2, 2, 1, 8, 4]))

sắp xếp: [1, 2, 2, 4, 4, 8] | cnt = {4: 2, 2: 2, 1: 1, 8: 1}
x=1: ghép (1, 2) | cnt={4: 2, 2: 1, 8: 1} | res=[1]
x=2: ghép (2, 4) | cnt={4: 1, 8: 1} | res=[1, 2]
x=2: cnt[x]=0 -> bỏ qua (đã dùng làm đôi)
x=4: ghép (4, 8) | cnt={} | res=[1, 2, 4]
x=4: cnt[x]=0 -> bỏ qua (đã dùng làm đôi)
x=8: cnt[x]=0 -> bỏ qua (đã dùng làm đôi)
Đáp án: [1, 2, 4]


In [16]:
print("Đáp án:", trace_counter([0, 0, 0, 1, 2, 3]))
print()
print("Đáp án:", trace_counter([6, 3, 0, 1]))

sắp xếp: [0, 0, 0, 1, 2, 3] | cnt = {0: 3, 1: 1, 2: 1, 3: 1}
x=0: ghép (0, 0) | cnt={0: 1, 1: 1, 2: 1, 3: 1} | res=[0]
x=0: không còn 0 để ghép -> []
Đáp án: []

sắp xếp: [0, 1, 3, 6] | cnt = {6: 1, 3: 1, 0: 1, 1: 1}
x=0: không còn 0 để ghép -> []
Đáp án: []


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def findOriginalArray(self, changed: List[int]) -> List[int]:
        if len(changed) % 2:              # mảng gấp đôi luôn có độ dài chẵn
            return []
        cnt = Counter(changed)
        res = []
        for x in sorted(changed):         # phần tử nhỏ nhất còn lại phải là gốc
            if cnt[x] == 0:               # đã được dùng làm "đôi" của số nhỏ hơn
                continue
            cnt[x] -= 1
            if cnt[2 * x] == 0:           # không tìm được đôi
                return []
            cnt[2 * x] -= 1
            res.append(x)
        return res

for ch, exp in examples:
    assert Solution().findOriginalArray(ch) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (lựa chọn bắt buộc).** Giả sử đa tập `S` là mảng gấp đôi của `O` và `m = min(S) > 0`. Trong cách ghép `S` thành các cặp `(o, 2o)`, `m` nằm trong một cặp nào đó. Nếu `m` là phần tử "đôi" `2o` thì `o = m/2 < m` cũng thuộc `S` — mâu thuẫn. Vậy `m` là phần tử gốc và cặp của nó là `(m, 2m)`. Với `m = 0`, cặp chứa `0` là `(0, 0)`.

**Bổ đề 2 (bất biến).** `S` là mảng gấp đôi ⇔ `S − {m, 2m}` là mảng gấp đôi (chiều ⇒ theo bổ đề 1; chiều ⇐ hiển nhiên, thêm cặp `(m, 2m)`).

**Kết luận.** Thuật toán lặp đúng bước bắt buộc này theo thứ tự tăng dần; nếu tại bước nào không có `2m`, theo bổ đề 1 không tồn tại cách ghép ⇒ `[]` đúng. Nếu hết phần tử, các `x` đã thêm vào `res` là một `original` hợp lệ. Hệ quả thú vị: **đa tập `original` là duy nhất** (chỉ thứ tự là tùy ý).

Trong Cách 3, "bỏ qua khi `cnt[x] == 0`" tương ứng việc phần tử đó đã bị lấy đi trong một cặp trước (với vai trò "đôi").

In [18]:
# Kiểm chứng "original là duy nhất (như đa tập)" bằng brute force: liệt kê MỌI tập con hợp lệ
def all_originals(changed):
    n = len(changed); res = set()
    if n % 2: return res
    for idx in combinations(range(n), n // 2):
        orig = [changed[i] for i in idx]
        if is_doubled_of(changed, orig):
            res.add(tuple(sorted(orig)))
    return res

rng = random.Random(10)
for _ in range(150):
    base = [rng.randint(0, 4) for _ in range(rng.randint(1, 5))]
    ch = base + [2 * x for x in base]; rng.shuffle(ch)
    assert len(all_originals(ch)) == 1
print("150 mảng gấp đôi ngẫu nhiên: luôn có đúng 1 đa tập original")

150 mảng gấp đôi ngẫu nhiên: luôn có đúng 1 đa tập original


## 11. Phân tích Time / Space

Gọi `n = len(changed)`, `V = max(changed) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force tổ hợp | `O(C(n, n/2) · n)` | `O(n)` |
| 2. Sắp xếp + `list.remove` | `O(n²)` | `O(n)` |
| 3. Sắp xếp + `Counter` | `O(n log n)` | `O(n)` |
| 4. Mảng đếm | `O(n + V)` | `O(V)` |
| 5. Sắp xếp + hàng đợi | `O(n log n)` | `O(n)` |

Đo trên `n = 10⁵`:

In [19]:
rng = random.Random(2007)
base = [rng.randint(0, 50000) for _ in range(50000)]
big = base + [2 * x for x in base]; rng.shuffle(big)
for name, fn in [("counter", find_counter), ("counting", find_counting), ("queue", find_queue)]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:8s}: len={len(r)}, hợp lệ={is_doubled_of(big, r)} ({time.perf_counter() - t:.3f}s)")
for m in (4000, 8000, 16000):
    small = base[:m // 2] + [2 * x for x in base[:m // 2]]; rng.shuffle(small)
    t = time.perf_counter(); r = find_greedy_list(small)
    print(f"list O(n^2) với n={m}: {time.perf_counter() - t:.3f}s (hợp lệ={is_doubled_of(small, r)})")

counter : len=50000, hợp lệ=True (0.075s)


counting: len=50000, hợp lệ=True (0.043s)


queue   : len=50000, hợp lệ=True (0.057s)
list O(n^2) với n=4000: 0.006s (hợp lệ=True)
list O(n^2) với n=8000: 0.023s (hợp lệ=True)


list O(n^2) với n=16000: 0.093s (hợp lệ=True)


## 12. Bộ test

Vì đáp án có thể theo thứ tự bất kỳ, ta kiểm: (a) nếu trả về khác rỗng thì phải **hợp lệ** theo `is_doubled_of`; (b) trả về rỗng **khi và chỉ khi** brute force cũng rỗng. Dữ liệu: ví dụ đề, biên (số 0, lặp, độ dài lẻ), mảng gấp đôi thật bị "làm hỏng" một phần tử, và ngẫu nhiên seed cố định.

In [20]:
fns = {
    "brute": find_brute, "list": find_greedy_list, "counter": find_counter,
    "counting": find_counting, "queue": find_queue,
    "Solution": lambda a: Solution().findOriginalArray(a),
    "SolutionCounting": lambda a: SolutionCounting().findOriginalArray(a),
}
def check(changed, fn, exists):
    r = fn(list(changed))
    if exists:
        return bool(r) and is_doubled_of(changed, r)
    return r == []

edge = [([0, 0], True), ([0], False), ([0, 0, 0, 0], True), ([0, 0, 0, 1, 2, 3], False),
        ([1, 2, 2, 4], True), ([2, 1, 2, 4, 4, 8], True), ([1, 1, 2, 2], True), ([1, 2, 4, 8, 16, 32], True),
        ([4, 8, 2, 16], True), ([100000, 50000], True), ([3, 6, 6, 12, 12, 24, 1], False)]
for ch, exp in examples:
    edge.append((ch, bool(exp)))
for ch, exists in edge:
    for name, fn in fns.items():
        assert check(ch, fn, exists), (name, ch)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [21]:
rng = random.Random(42)
pos = neg = 0
for _ in range(400):
    base = [rng.randint(0, 6) for _ in range(rng.randint(1, 6))]
    ch = base + [2 * x for x in base]
    if rng.random() < 0.5:                       # làm hỏng một phần tử
        i = rng.randrange(len(ch)); ch[i] = rng.randint(0, 12)
    rng.shuffle(ch)
    exists = bool(find_brute(ch))
    pos += exists; neg += not exists
    for name, fn in fns.items():
        assert check(ch, fn, exists), (name, ch)
print(f"400 test ngẫu nhiên ({pos} có đáp án, {neg} không): mọi cách khớp brute force")

400 test ngẫu nhiên (237 có đáp án, 163 không): mọi cách khớp brute force


In [22]:
rng = random.Random(77)
for _ in range(50):
    base = [rng.randint(0, 1000) for _ in range(rng.randint(1, 2000))]
    ch = base + [2 * x for x in base]; rng.shuffle(ch)
    if rng.random() < 0.5:
        ch[rng.randrange(len(ch))] += 1
    r3, r4, r5 = find_counter(ch), find_counting(ch), find_queue(ch)
    assert sorted(r3) == sorted(r4) == sorted(r5)
    assert (not r3) or is_doubled_of(ch, r3)
print("50 test cỡ lớn: các cách nhanh khớp nhau")

50 test cỡ lớn: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp, ghép theo thứ tự xuất hiện

Với `[4, 8, 2, 16]` (gốc là `[2, 8]`), nếu ghép `4` với `8` trước thì `2` không còn `4`.

In [23]:
def wrong_no_sort(changed):
    if len(changed) % 2: return []
    cnt = Counter(changed); res = []
    for x in changed:                      # SAI: không sắp xếp
        if cnt[x] == 0: continue
        cnt[x] -= 1
        if cnt[2 * x] == 0: return []
        cnt[2 * x] -= 1; res.append(x)
    return res

ch = [4, 8, 2, 16]
print("sai:", wrong_no_sort(ch), "| đúng:", Solution().findOriginalArray(ch))

sai: [] | đúng: [2, 8]


In [24]:
print("sửa (duyệt sorted):", find_counter([4, 8, 2, 16]))

sửa (duyệt sorted): [2, 8]


### Lỗi 2 — Dùng `set` thay cho đếm, mất thông tin số lần lặp

`[1, 1, 2, 2]` có gốc `[1, 1]`. Dùng `set` thì chỉ còn `{1, 2}`.

In [25]:
def wrong_set(changed):
    if len(changed) % 2: return []
    s = set(changed); res = []
    for x in sorted(s):                    # SAI: mất số lần lặp
        if x in s and 2 * x in s:
            s.discard(x); s.discard(2 * x); res.append(x)
    return res

ch = [1, 1, 2, 2]
r = wrong_set(ch)
print("sai:", r, "hợp lệ?", is_doubled_of(ch, r), "| đúng:", Solution().findOriginalArray(ch))

sai: [1] hợp lệ? False | đúng: [1, 1]


In [26]:
print("sửa (Counter):", find_counter([1, 1, 2, 2]))

sửa (Counter): [1, 1]


### Lỗi 3 — Xử lý nhóm nhưng quên trường hợp `x = 0`

Khi xử lý cả nhóm `k = cnt[x]` một lúc, với `x = 0` thì "đôi" của `0` cũng là `0`: không thể dùng công thức `cnt[2x] -= k` (sẽ trừ chính nó) mà phải lấy `cnt[0] // 2`.

In [27]:
def wrong_zero_group(changed):
    if len(changed) % 2: return []
    cnt = Counter(changed); res = []
    for x in sorted(cnt):
        k = cnt[x]
        if k == 0: continue
        if cnt[2 * x] < k: return []
        cnt[2 * x] -= k                    # SAI khi x = 0: trừ vào chính cnt[0]
        res.extend([x] * k)
    return res

ch = [0, 0]
r = wrong_zero_group(ch)
print("sai:", r, "hợp lệ?", is_doubled_of(ch, r), "| đúng:", Solution().findOriginalArray(ch))

sai: [0, 0] hợp lệ? False | đúng: [0]


In [28]:
print("sửa (xử lý 0 riêng):", find_counting([0, 0]), "| [0,0,0,0] ->", find_counting([0, 0, 0, 0]))

sửa (xử lý 0 riêng): [0] | [0,0,0,0] -> [0, 0]


### Lỗi 4 — Cách hàng đợi: quên kiểm tra hàng đợi rỗng ở cuối

Nếu kết thúc vòng lặp mà vẫn còn giá trị "đang chờ" (ví dụ `[1, 3]`: chờ `2` và `6` mãi không tới) thì không phải mảng gấp đôi. Bản sai dưới đây bỏ cả kiểm tra độ dài lẻ lẫn kiểm tra cuối.

In [29]:
def wrong_queue(changed):
    pending, res = deque(), []
    for y in sorted(changed):
        if pending and pending[0] == y:
            pending.popleft()
        else:
            res.append(y); pending.append(2 * y)
    return res                               # SAI: không kiểm pending rỗng

for ch in ([1], [1, 3], [6, 3, 0, 1]):
    print(ch, "sai:", wrong_queue(ch), "| đúng:", Solution().findOriginalArray(ch))

[1] sai: [1] | đúng: []
[1, 3] sai: [1, 3] | đúng: []
[6, 3, 0, 1] sai: [0, 1, 3, 6] | đúng: []


In [30]:
print("sửa:", [find_queue(ch) for ch in ([1], [1, 3], [6, 3, 0, 1])])

sửa: [[], [], []]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ghép cặp (x, f(x)) trong đa tập"** với `f` tăng (ở đây `f(x) = 2x`) ⇒ **sắp xếp** và xử lý từ nhỏ nhất: phần tử nhỏ nhất chỉ có thể đóng vai "trái".
- Đa tập ⇒ `Counter`; giá trị nhỏ (`≤ 10⁵`) ⇒ có thể **mảng đếm** cho `O(n + V)`.
- Luôn kiểm **điểm bất động** của `f` (ở đây `f(0) = 0`).
- Kiểm độ dài lẻ ngay đầu hàm — vừa đúng vừa nhanh.
- Bài trả về "thứ tự bất kỳ" ⇒ viết hàm kiểm hợp lệ khi test.

## 15. Biến thể của bài

1. **LeetCode 954 — Array of Doubled Pairs:** có số **âm**; chỉ hỏi có ghép được không. Sắp theo **giá trị tuyệt đối** (với số âm, "nhỏ nhất" theo |x|).
2. **Ghép `(x, x + k)`**: cùng tham lam, `f(x) = x + k` (`k > 0`).
3. **LeetCode 846 — Hand of Straights:** chia thành các nhóm `W` số liên tiếp — tham lam từ nhỏ nhất với `Counter`.
4. **Mảng "gấp ba"** hoặc **gấp `k`**: thay `2x` bằng `k·x`.

Minh họa biến thể 1 (tự dựng, kiểm bằng brute force có số âm):

In [31]:
def can_reorder_doubled(arr):
    cnt = Counter(arr)
    for x in sorted(cnt, key=abs):
        if cnt[x] == 0: continue
        if x == 0:
            if cnt[0] % 2: return False
            cnt[0] = 0; continue
        if cnt[2 * x] < cnt[x]: return False
        cnt[2 * x] -= cnt[x]; cnt[x] = 0
    return True

rng = random.Random(954)
for _ in range(300):
    arr = [rng.randint(-5, 5) for _ in range(rng.choice([2, 4, 6]))]
    n = len(arr)
    brute = any(Counter(2 * arr[i] for i in idx) == Counter(arr[i] for i in range(n) if i not in idx)
                for idx in combinations(range(n), n // 2))
    assert can_reorder_doubled(arr) == brute
print("Biến thể có số âm khớp brute force; [4,-2,2,-4] ->", can_reorder_doubled([4, -2, 2, -4]))

Biến thể có số âm khớp brute force; [4,-2,2,-4] -> True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi tập con n/2 | hàm mũ | `O(n)` | kiểm tra, `n ≤ 14` |
| 2. Sort + `list.remove` | tham lam ngây thơ | `O(n²)` | `O(n)` | minh họa ý tưởng |
| 3. Sort + `Counter` | tham lam từ nhỏ nhất | `O(n log n)` | `O(n)` | **nộp bài**, tổng quát |
| 4. Mảng đếm | duyệt giá trị 0..V | `O(n + V)` | `O(V)` | giá trị nhỏ, cần nhanh nhất |
| 5. Sort + hàng đợi | đôi đang chờ, không băm | `O(n log n)` | `O(n)` | khi không muốn dùng hash |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `findOriginalArray([2, 4, 1, 2])` (nêu đa tập).
2. **Tính tay:** trace Cách 5 (hàng đợi) trên `[1, 2, 2, 4, 4, 8]`.
3. **Điền TODO:** hoàn thiện `is_doubled(changed)` chỉ trả `True/False`.
4. **Tự viết lại:** viết lại Cách 4 (mảng đếm) không nhìn code.
5. **Mini challenge:** `find_original_k(changed, k)` cho mảng "gấp `k`" (`k ≥ 2`, giá trị `≥ 0`). Kiểm bằng brute force.

In [32]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_doubled(changed):
    # TODO
    pass

In [33]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_counting(changed):
    # TODO: mảng đếm, xử lý 0 riêng, duyệt x = 1..V
    pass

In [34]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def find_original_k(changed, k):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Sắp: `[1, 2, 2, 4]`. (2) `1` ghép `2`. (3) `2` ghép `4` ⇒ `[1, 2]`.

**Bài 2.** (1) `pending` rỗng ⇒ `1` là gốc, đẩy `2`. (2) `2 == pending[0]` ⇒ lấy ra; `2` tiếp theo là gốc, đẩy `4`. (3) Tiếp tục với `4, 4, 8`.

**Bài 3.** (1) Tái sử dụng `find_counter`. (2) Cẩn thận: `[0, 0]` có đáp án `[0]` (khác rỗng). (3) `return len(changed) % 2 == 0 and bool(find_counter(changed))`.

**Bài 4.** (1) `cnt = [0] * (V + 1)`. (2) `cnt[0]` lẻ ⇒ `[]`. (3) Với `x > 0`, kiểm `2x ≤ V` trước khi truy cập `cnt[2x]`.

**Bài 5.** (1) Như Cách 3, thay `2*x` bằng `k*x`. (2) `x = 0` vẫn ghép với `0`. (3) Trừ `cnt[x]` trước khi kiểm `cnt[k*x]`.

## 19. Đáp án tham khảo

In [35]:
# Bài 1
r = Solution().findOriginalArray([2, 4, 1, 2])
assert sorted(r) == [1, 2]
print("Bài 1:", r)
# Bài 2: trace hàng đợi
def trace_queue(changed):
    pending, res = deque(), []
    for y in sorted(changed):
        if pending and pending[0] == y:
            pending.popleft(); act = "là đôi -> lấy khỏi hàng đợi"
        else:
            res.append(y); pending.append(2 * y); act = f"là gốc -> chờ {2 * y}"
        print(f"y={y}: {act} | pending={list(pending)} | res={res}")
    return res if not pending else []
print("Bài 2:", trace_queue([1, 2, 2, 4, 4, 8]))

Bài 1: [1, 2]
y=1: là gốc -> chờ 2 | pending=[2] | res=[1]
y=2: là đôi -> lấy khỏi hàng đợi | pending=[] | res=[1]
y=2: là gốc -> chờ 4 | pending=[4] | res=[1, 2]
y=4: là đôi -> lấy khỏi hàng đợi | pending=[] | res=[1, 2]
y=4: là gốc -> chờ 8 | pending=[8] | res=[1, 2, 4]
y=8: là đôi -> lấy khỏi hàng đợi | pending=[] | res=[1, 2, 4]
Bài 2: [1, 2, 4]


In [36]:
# Bài 3, 4, 5
def is_doubled_ans(changed):
    return len(changed) % 2 == 0 and bool(find_counter(changed))

def my_counting_ans(changed):
    return find_counting(changed)

def find_original_k_ans(changed, k):
    if len(changed) % 2: return []
    cnt = Counter(changed); res = []
    for x in sorted(changed):
        if cnt[x] == 0: continue
        cnt[x] -= 1
        if cnt[k * x] == 0: return []
        cnt[k * x] -= 1; res.append(x)
    return res

def brute_k(changed, k):
    n = len(changed)
    if n % 2: return False
    return any(Counter(k * changed[i] for i in idx) == Counter(changed[i] for i in range(n) if i not in idx)
               for idx in combinations(range(n), n // 2))

rng = random.Random(6)
for _ in range(300):
    k = rng.randint(2, 4)
    base = [rng.randint(0, 5) for _ in range(rng.randint(1, 5))]
    ch = base + [k * x for x in base]
    if rng.random() < 0.5: ch[rng.randrange(len(ch))] = rng.randint(0, 20)
    rng.shuffle(ch)
    r = find_original_k_ans(ch, k)
    assert bool(r) == brute_k(ch, k)
    assert not r or Counter(r + [k * x for x in r]) == Counter(ch)
    if k == 2:
        assert is_doubled_ans(ch) == brute_k(ch, 2)
        assert sorted(my_counting_ans(ch)) == sorted(find_counter(ch))
print("Bài 3, 4, 5: OK; ví dụ gấp 3:", find_original_k_ans([3, 1, 9, 3], 3))

Bài 3, 4, 5: OK; ví dụ gấp 3: [1, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ghép cặp (x, f(x)) tham lam từ nhỏ nhất":**

```python
if len(a) % 2: return []
cnt = Counter(a); res = []
for x in sorted(a):
    if cnt[x] == 0: continue        # đã là "đôi" của ai đó
    cnt[x] -= 1                     # trừ trước (đúng cả khi f(x) == x)
    if cnt[f(x)] == 0: return []
    cnt[f(x)] -= 1; res.append(x)
return res
```

In [37]:
def template(a, f=lambda x: 2 * x):
    if len(a) % 2: return []
    cnt = Counter(a); res = []
    for x in sorted(a):
        if cnt[x] == 0: continue
        cnt[x] -= 1
        if cnt[f(x)] == 0: return []
        cnt[f(x)] -= 1; res.append(x)
    return res
print([template(ch) for ch, _ in examples])

[[1, 3, 4], [], []]


**Checklist:**

- [ ] Độ dài lẻ ⇒ `[]` ngay.
- [ ] **Sắp xếp** (hoặc duyệt giá trị tăng dần) trước khi ghép.
- [ ] Dùng **đếm** (`Counter`/mảng), không dùng `set`.
- [ ] Số `0`: cần số lượng chẵn; trừ `cnt[x]` trước khi kiểm `cnt[2x]`.
- [ ] Mảng đếm: kiểm `2x ≤ V` trước khi truy cập.
- [ ] Test bằng hàm hợp lệ vì thứ tự đáp án tùy ý.